# FIFA World Cup 2026 — Notebook 04: Tournament Fixtures

## About

**Purpose:** Define the 48 qualified teams, their 12 groups, and the group-stage fixture list.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-07<br>
**Notes:** The real World Cup 2026 draw (5 Dec 2025) — 12 groups of 4. Team names are reconciled to match the `team_strengths` index from notebook 02 exactly, then every team is validated to exist before saving. Group fixtures are the full round-robin (6 matches per group, 72 total). The knockout bracket is built in notebook 05, where it is simulated. Phase 0 ignores host advantage (USA/Canada/Mexico).<br>
**Description:** Produces `wc_groups.parquet` (team → group) and `wc_group_fixtures.parquet` (group, home_team, away_team) for the Monte Carlo simulation in notebook 05.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-06-07 | 1.0     | Ganapathy K | Initial version |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import pandas as pd
from itertools import combinations
from pathlib import Path

### 1.2 Config

In [3]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
TEAM_STRENGTHS_PATH = PROCESSED_DATA_DIR / "team_strengths.parquet"
GROUPS_PATH = PROCESSED_DATA_DIR / "wc_groups.parquet"
GROUP_FIXTURES_PATH = PROCESSED_DATA_DIR / "wc_group_fixtures.parquet"


## 2. The 12 Groups

The official draw, written with the team names exactly as FIFA published them. A few of those differ from the spelling in our results dataset, so `NAME_MAP` rewrites them to the dataset's version (e.g. "Czechia" → "Czech Republic"). Everything not in the map is already an exact match.

In [4]:
draw = {
    "A": ["Mexico", "South Africa", "South Korea", "Czechia"],
    "B": ["Canada", "Bosnia-Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkiye"],
    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

NAME_MAP = {
    "Czechia": "Czech Republic",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Turkiye": "Turkey",
    "Curacao": "Curaçao",
    "Congo DR": "DR Congo",
}

groups = {g: [NAME_MAP.get(t, t) for t in teams] for g, teams in draw.items()}
groups

{'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
 'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
 'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
 'D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
 'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
 'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
 'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
 'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
 'I': ['France', 'Senegal', 'Iraq', 'Norway'],
 'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
 'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
 'L': ['England', 'Croatia', 'Ghana', 'Panama']}

## 3. Validate Against Strengths

Every one of the 48 teams must have a strength row, or the Poisson engine will `KeyError` mid-simulation. Check now, fail loud here.

In [5]:
team_strengths = pd.read_parquet(TEAM_STRENGTHS_PATH)
known_teams = set(team_strengths.index)

all_teams = [team for teams in groups.values() for team in teams]
missing = [team for team in all_teams if team not in known_teams]

assert len(all_teams) == 48, f"Expected 48 teams, got {len(all_teams)}"
assert not missing, f"Teams with no strength row: {missing}"
print(f"All {len(all_teams)} teams validated against strengths table.")

All 48 teams validated against strengths table.


## 4. Group-Stage Fixtures

Within each group every team plays every other once — `combinations(teams, 2)` gives the 6 pairings per group, 72 matches total. Home/away here is just an ordering label; phase 0 has no home-advantage term, and World Cup venues are neutral anyway.

In [6]:
fixtures = []
for group, teams in groups.items():
    for home_team, away_team in combinations(teams, 2):
        fixtures.append({"group": group, "home_team": home_team, "away_team": away_team})

group_fixtures = pd.DataFrame(fixtures)
print(f"Total group-stage matches: {len(group_fixtures)}")
group_fixtures.head(8)

Total group-stage matches: 72


,group,home_team,away_team
0,A,Mexico,South Africa
1,A,Mexico,South Korea
2,A,Mexico,Czech Republic
3,A,South Africa,South Korea
4,A,South Africa,Czech Republic
5,A,South Korea,Czech Republic
6,B,Canada,Bosnia and Herzegovina
7,B,Canada,Qatar


## 5. Save

Write a tidy team→group table and the fixture list for notebook 05.

In [7]:
groups_table = pd.DataFrame(
    [{"team": team, "group": group} for group, teams in groups.items() for team in teams]
)

groups_table.to_parquet(GROUPS_PATH, index=False)
group_fixtures.to_parquet(GROUP_FIXTURES_PATH, index=False)
print(f"Saved {len(groups_table)} teams -> {GROUPS_PATH.name}")
print(f"Saved {len(group_fixtures)} fixtures -> {GROUP_FIXTURES_PATH.name}")

Saved 48 teams -> wc_groups.parquet
Saved 72 fixtures -> wc_group_fixtures.parquet
